# 02 - Preprocessing & Feature Engineering

Questa fase trasforma i dati grezzi in feature pronte per il Machine Learning.

**Step:**
1. Caricamento dati puliti da 01_Data_Ingestion_EDA
2. Data Quality Check (duplicati, missing values)
3. Feature Extraction (Coupon, DTM, YTM)
4. Gestione delle frequenze miste (Forward-Fill, Shift per Lookahead Bias)
5. Creazione Lagged Features per Time-Series
6. Merge e allineamento temporale finale

## 2.1 - Import & Load Cleaned Data

In [78]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Carica i dataset puliti dal notebook precedente
df_anagrafica_clean = pd.read_csv('./data/df_anagrafica_clean.csv')
df_storico_clean = pd.read_csv('./data/df_storico_clean.csv')
df_macro = pd.read_csv('./data/df_macro.csv', index_col=0, parse_dates=True)

# Assicurati che le date siano datetime
df_storico_clean['referencedate'] = pd.to_datetime(df_storico_clean['referencedate'])
df_anagrafica_clean['redemptiondate'] = pd.to_datetime(df_anagrafica_clean['redemptiondate'], format='%d/%m/%Y', errors='coerce')

print(f"✓ Dataset caricati")
print(f"  Anagrafica: {df_anagrafica_clean.shape}")
print(f"  Storico: {df_storico_clean.shape}")
print(f"  Macro: {df_macro.shape}")

✓ Dataset caricati
  Anagrafica: (305, 10)
  Storico: (189510, 9)
  Macro: (9485, 12)


## 2.2 - Data Quality Check

In [79]:
# 1. Duplicati su ISIN+Data
if 'isincode' in df_storico_clean.columns and 'referencedate' in df_storico_clean.columns:
    n_dupes = df_storico_clean.duplicated(subset=['isincode', 'referencedate']).sum()
    print(f"Duplicati su ISIN+Data nello storico: {n_dupes}")
    
# 2. Missing nelle colonne chiave
print("\nMissing values nelle colonne chiave:")
print(df_storico_clean[['isincode', 'referencedate', 'pricevalue']].isnull().sum())
print("\nAnag missing:")
print(df_anagrafica_clean[['isincode', 'redemptiondate', 'description']].isnull().sum())

Duplicati su ISIN+Data nello storico: 0

Missing values nelle colonne chiave:
isincode         0
referencedate    0
pricevalue       0
dtype: int64

Anag missing:
isincode          0
redemptiondate    0
description       0
dtype: int64


## 2.3 - Feature Extraction: Coupon (Text Mining)

In [80]:
def extract_coupon(description):
    """
    Estrae il tasso di cedola (coupon) dalla descrizione del titolo.
    
    - Zero Coupon: Ritorna 0.0 se contiene 'bot', 'zc', 'zero'
    - Pattern numerico: Cerca "5%" o "3,5%" nella descrizione
    - Pattern EUR: Cerca "EUR 5" nella descrizione
    
    Returns: float (%) oppure np.nan
    """
    if pd.isnull(description):
        return np.nan
    desc = str(description)

    # ZERO COUPON
    if 'bot' in desc.lower() or 'zc' in desc.lower() or 'zero' in desc.lower() or 'ctz' in desc.lower():
        return 0.0

    # Pattern: numeri + %
    match = re.search(r'(\d+[\.,]\d+|\d+)[ ]*%', desc)
    if match:
        return float(match.group(1).replace(',', '.'))
    
    # Pattern: EUR + numeri
    match = re.search(r'[Ee][Uu][Rr][ ]*(\d+[\.,]\d+|\d+)', desc)
    if match:
        return float(match.group(1).replace(',', '.'))
   
    return np.nan

df_anagrafica_clean['coupon'] = df_anagrafica_clean['description'].apply(extract_coupon)

print("--- COUPON EXTRACTION RESULTS ---")
print(f"Coupon estratti: {df_anagrafica_clean['coupon'].notna().sum()}")
print(f"Coupon mancanti: {df_anagrafica_clean['coupon'].isna().sum()}")
print(f"\nCoupon Statistics:")
print(df_anagrafica_clean['coupon'].describe())

# Visualizza alcuni esempi
print("\nEsempi di estrazione:")
display(df_anagrafica_clean[['description', 'coupon']].head(10))

--- COUPON EXTRACTION RESULTS ---
Coupon estratti: 305
Coupon mancanti: 0

Coupon Statistics:
count    305.000000
mean       2.327951
std        1.663578
min        0.000000
25%        0.850000
50%        2.500000
75%        3.450000
max        7.250000
Name: coupon, dtype: float64

Esempi di estrazione:


,description,coupon
0,Btp Fx 3.15% Jun31 Eur,3.15
1,Schatz Fx 2.5% Jun28 Eur,2.50
2,Btp Fx 3.8% Jul36 Eur,3.80
3,Btp Fx 3.3% Jun33 Eur,3.30
4,Bot Zc Apr27 A Eur,0.00
5,Bot Zc Jul26 Q Eur,0.00
6,Bot Zc Sep26 S Eur,0.00
7,Bot Zc Mar27 A Eur,0.00
8,Obligaciones Fx 3.95% Oct56 Eur,3.95
9,Bonos Fx 2.6% May31 Eur,2.60
